# Finite Element Method for a 1D Beam Problem

Start with a simple 1D beam problem.

A beam with 5 nodes (A-E) connect by 4 springs (1-4) with one end fixed and the other end being pulled by external force F

```text
////|
////|
////A --- B --- C --- D --- E  => F
////|  1     2     3     4 
////|
```

Problem: Find the displacement of each node.

We will assume each node is connected together by a spring of stiffness k.

5 nodes => 5 degrees of freedom (DOF)

Fundamental equation of FEM: `K * u = F` where
 - `K` is the stiffness matrix (5x5)
 - `u` is the displacement vector (5x1)
 - `F` is the external force vector (5x1)

The global stiffness matrix will look like this:

```text
[  k_1, -k_1      ,           ,           ,      ]
[ -k_1, k_1 + k_2 , -k_2      ,           ,      ]
[     , -k_2      , k_3 + k_2 , -k_3      ,      ]
[     ,           , -k_3      , k_4 + k_3 , -k_4 ]
[     ,           ,           , -k_4      , k_4  ]
```

By stating the inital displacements

```text
u = [0, u_b, u_c, u_d, u_e]
```

And the external force vector
```text
F = [0, 0, 0, 0, F]
```

We can solve for the displacements of each node.

If `K * u = F` Then `u = K^-1 * F`

In [ ]:
import numpy as np

def make_stiffness_matrix(k_1, k_2, k_3, k_4):
    return np.array([
        [k_1, -k_1, 0, 0, 0],
        [-k_1, k_1 + k_2, -k_2, 0, 0],
        [0, -k_2, k_2 + k_3, -k_3, 0],
        [0, 0, -k_3, k_3 + k_4, -k_4],
        [0, 0, 0, -k_4, k_4]
    ])

def make_force_vector(F):
    return np.array([0, 0, 0, 0, 1])

STIFFNESS = 5.0
EXTERNAL_FORCE = 10.0

K = make_stiffness_matrix(STIFFNESS, STIFFNESS, STIFFNESS, STIFFNESS)
F = make_force_vector(EXTERNAL_FORCE)

# Boundary condition: node 0 is fixed
K[0, :] = 0
K[:, 0] = 0
K[0, 0] = 1

K_inv = np.linalg.inv(K)
u = K_inv @ F
u
# 

# 2D Example FEM

We will employ a similar approach to solve a 2D beam problem.

```text
////|
////|
////|                F
////|                |
////|                V
////A ------ C ----- D
////||      -/
////||    -/
////||  -/
////|| /
////B
////|
////|
////|
```

In [ ]:
import matplotlib.pyplot as plt

class Node:
    def __init__(self, node_id, x, y):
        self.node_id = node_id
        self.x = x
        self.y = y

class Element:
    def __init__(self, node1, node2):
        self.node1 = node1
        self.node2 = node2

class Mesh:
    def __init__(self, nodes, elements):
        self.nodes = nodes
        self.elements = elements
        self.fixed_dofs = []

    def fix_dof(self, node_index, dof):
        self.fixed_dofs.append((node_index * 3) + dof)

    def draw(self, u=None, line_color="black", node_color="red"):
        if u is None:
            u = np.zeros(len(self.nodes) * 3)
        for element in self.elements:
            u1x = u[(element.node1.node_id * 3)]
            u2x = u[(element.node2.node_id * 3)]
            u1y = u[(element.node1.node_id * 3) + 1]
            u2y = u[(element.node2.node_id * 3) + 1]
            plt.plot([element.node1.x + u1x, element.node2.x + u2x], [element.node1.y + u1y, element.node2.y + u2y], c=line_color)

        for node in self.nodes:
            ux = u[(node.node_id * 3)]
            uy = u[(node.node_id * 3) + 1]
            plt.scatter([node.x + ux], [node.y + uy], color=node_color)

nodes = [
    # Bottom chord / deck nodes
    Node(0, 0, 0),    # A
    Node(1, 2, 0),    # B
    Node(2, 4, 0),    # C
    Node(3, 6, 0),    # D
    Node(4, 8, 0),    # E
    Node(5, 10, 0),   # F
    Node(6, 12, 0),   # G

    # Top chord nodes
    Node(7, 0, 1.0),    # H
    Node(8, 2, 2.0),    # I
    Node(9, 4, 2.8),    # J
    Node(10, 6, 3.2),   # K
    Node(11, 8, 2.8),   # L
    Node(12, 10, 2.0),  # M
    Node(13, 12, 1.0),  # N
]

element_pairs = [
    # Bottom chord
    (0, 1),
    (1, 2),
    (2, 3),
    (3, 4),
    (4, 5),
    (5, 6),

    # Top chord
    (7, 8),
    (8, 9),
    (9, 10),
    (10, 11),
    (11, 12),
    (12, 13),

    # Verticals
    (0, 7),
    (1, 8),
    (2, 9),
    (3, 10),
    (4, 11),
    (5, 12),
    (6, 13),

    # Diagonal web members
    (0, 8),
    (1, 7),
    (1, 9),
    (2, 8),
    (2, 10),
    (3, 9),
    (3, 11),
    (4, 10),
    (4, 12),
    (5, 11),
    (5, 13),
    (6, 12),
]

elements = [
    Element(nodes[i], nodes[j])
    for i, j in element_pairs
]

mesh = Mesh(nodes, elements)

# Left support: pinned
for n in (0,6):
    for dof in (0, 1, 2):
        mesh.fix_dof(n, dof)

mesh.draw()

plt.xlim(-1, 13)
plt.ylim(-1, 4)
plt.gca().set_aspect("equal", adjustable="box")
plt.title("Arched Warren Frame")

In [ ]:
import numpy as np

FLEXUAL_RIGIDITY = 1.0

def get_local_stiffness_matrix(element):
    dx = element.node2.x - element.node1.x
    dy = element.node2.y - element.node1.y
    L = np.sqrt(dx**2 + dy**2)

    return np.array([
        [ 1/L,        0,          0, -1/L,        0,          0],
        [   0,  12/L**3,   6/L**2,    0, -12/L**3,   6/L**2],
        [   0,   6/L**2,     4/L,    0,  -6/L**2,     2/L],
        [-1/L,        0,          0,  1/L,        0,          0],
        [   0, -12/L**3,  -6/L**2,    0,  12/L**3,  -6/L**2],
        [   0,   6/L**2,     2/L,    0,  -6/L**2,     4/L],
    ], dtype=float)

# We need to transform the local matrix to the global space before adding them up
def get_element_transformation_matrix(element):
    node1 = element.node1
    node2 = element.node2

    # compute the angle of the element
    dx = node2.x - node1.x
    dy = node2.y - node1.y
    theta = np.arctan2(dy, dx) # In radians

    rotation_matrix = np.array([
        [np.cos(theta), np.sin(theta)],
        [-np.sin(theta), np.cos(theta)]
    ])

    full_matrix = np.zeros(shape=(6,6))
    full_matrix[0:2,0:2] = rotation_matrix
    full_matrix[3:5,3:5] = rotation_matrix
    full_matrix[2,2] = 1
    full_matrix[5,5] = 1

    return full_matrix

T = get_element_transformation_matrix(elements[1])
K = get_local_stiffness_matrix(elements[1])

In [ ]:
def construct_global_stiffness_matrix(mesh):
    node_count = len(mesh.nodes)
    degrees_of_freedom = node_count * 3
    stiffness_matrix = np.zeros((degrees_of_freedom,degrees_of_freedom))

    for e in mesh.elements:
        K_e = get_local_stiffness_matrix(e)
        T = get_element_transformation_matrix(e)

        K = T.T @ K_e @ T
        node_1_id = e.node1.node_id
        node_2_id = e.node2.node_id

        stiffness_matrix[node_1_id*3:(node_1_id+1)*3,node_1_id*3:(node_1_id+1)*3] += K[0:3,0:3]
        stiffness_matrix[node_2_id*3:(node_2_id+1)*3,node_1_id*3:(node_1_id+1)*3] += K[3:6,0:3]
        stiffness_matrix[node_1_id*3:(node_1_id+1)*3,node_2_id*3:(node_2_id+1)*3] += K[0:3,3:6]
        stiffness_matrix[node_2_id*3:(node_2_id+1)*3,node_2_id*3:(node_2_id+1)*3] += K[3:6,3:6]

    return stiffness_matrix

def fix_dof(M, dof):
    M[dof,:] = 0
    M[:,dof] = 0
    M[dof,dof] = 1
    return M

K = construct_global_stiffness_matrix(mesh)
for i in mesh.fixed_dofs:
    K = fix_dof(K, i)

In [ ]:
F = np.zeros((len(mesh.nodes) * 3))
F[(10 * 3) + 1] = -0.3

In [ ]:
K_inv = np.linalg.inv(K)
u = K_inv @ F
u

In [ ]:
mesh.draw(line_color="#AAAAAA", node_color="blue")
mesh.draw(u)
plt.xlim(-1, 13)
plt.ylim(-1, 4)
plt.gca().set_aspect("equal", adjustable="box")
plt.title("Arched Warren Frame")

In [ ]:
stiffness = []
for element in mesh.elements:
    dx = element.node1.x - element.node2.x
    dy = element.node1.y - element.node2.y
    L = np.sqrt(dx**2 + dy**2)

    dxu = dx - u[(element.node1.node_id * 3)] + u[(element.node2.node_id * 3)]
    dyu = dy - u[(element.node1.node_id * 3)+1] + u[(element.node2.node_id * 3)+1]
    Lu = np.sqrt(dxu**2 + dyu**2)

    axial_load = (Lu - L)
    stiffness.append(axial_load)

stiffness = (np.array(stiffness) - min(stiffness)) / (max(stiffness) - min(stiffness))

plt.xlim(-1, 13)
plt.ylim(-1, 4)
plt.gca().set_aspect("equal", adjustable="box")
plt.title("Arched Warren Frame Axial Extenstions")

for i, element in enumerate(mesh.elements):
    u1x = u[(element.node1.node_id * 3)]
    u2x = u[(element.node2.node_id * 3)]
    u1y = u[(element.node1.node_id * 3) + 1]
    u2y = u[(element.node2.node_id * 3) + 1]
    cmap = plt.colormaps['plasma']
    plt.plot([element.node1.x + u1x, element.node2.x + u2x], [element.node1.y + u1y, element.node2.y + u2y], c=cmap(float(stiffness[i])))

# Dynamics Structures using Newmark-Beta Method

$$
Ma(t) + Cv(t)+Ku(t) = f_{ext}(t)
$$

 - M - resistance to acceleration
 - C - Energy Dissipation
 - K - resistance to deformation

In [ ]:
import matplotlib.pyplot as plt

class Node:
    def __init__(self, node_id, x, y):
        self.node_id = node_id
        self.x = x
        self.y = y

class Element:
    def __init__(self, node1, node2):
        self.node1 = node1
        self.node2 = node2

class Mesh:
    def __init__(self, nodes, elements):
        self.nodes = nodes
        self.elements = elements
        self.fixed_dofs = []

    def fix_dof(self, node_index, dof):
        self.fixed_dofs.append((node_index * 3) + dof)

    def draw(self, u=None, line_color="black", node_color="red"):
        if u is None:
            u = np.zeros(len(self.nodes) * 3)
        for element in self.elements:
            u1x = u[(element.node1.node_id * 3)]
            u2x = u[(element.node2.node_id * 3)]
            u1y = u[(element.node1.node_id * 3) + 1]
            u2y = u[(element.node2.node_id * 3) + 1]
            plt.plot([element.node1.x + u1x, element.node2.x + u2x], [element.node1.y + u1y, element.node2.y + u2y], c=line_color)

        for node in self.nodes:
            ux = u[(node.node_id * 3)]
            uy = u[(node.node_id * 3) + 1]
            plt.scatter([node.x + ux], [node.y + uy], color=node_color)

nodes = [
    # Bottom chord / deck nodes
    Node(0, 0, 0),    # A
    Node(1, 2, 0),    # B
    Node(2, 4, 0),    # C
    Node(3, 6, 0),    # D
    Node(4, 8, 0),    # E
    Node(5, 10, 0),   # F
    Node(6, 12, 0),   # G

    # Top chord nodes
    Node(7, 0, 1.0),    # H
    Node(8, 2, 2.0),    # I
    Node(9, 4, 2.8),    # J
    Node(10, 6, 3.2),   # K
    Node(11, 8, 2.8),   # L
    Node(12, 10, 2.0),  # M
    Node(13, 12, 1.0),  # N
]

element_pairs = [
    # Bottom chord
    (0, 1),
    (1, 2),
    (2, 3),
    (3, 4),
    (4, 5),
    (5, 6),

    # Top chord
    (7, 8),
    (8, 9),
    (9, 10),
    (10, 11),
    (11, 12),
    (12, 13),

    # Verticals
    (0, 7),
    (1, 8),
    (2, 9),
    (3, 10),
    (4, 11),
    (5, 12),
    (6, 13),

    # Diagonal web members
    (0, 8),
    (1, 7),
    (1, 9),
    (2, 8),
    (2, 10),
    (3, 9),
    (3, 11),
    (4, 10),
    (4, 12),
    (5, 11),
    (5, 13),
    (6, 12),
]

elements = [
    Element(nodes[i], nodes[j])
    for i, j in element_pairs
]

mesh = Mesh(nodes, elements)

# Left support: pinned
for n in (0,6):
    for dof in (0, 1, 2):
        mesh.fix_dof(n, dof)

mesh.draw()

plt.xlim(-1, 13)
plt.ylim(-1, 4)
plt.gca().set_aspect("equal", adjustable="box")
plt.title("Arched Warren Frame")

In [ ]:
# Probably a poor approximation
def get_mass_matrix(mesh, density):
    mass_matrix = np.zeros((len(mesh.nodes) * 3, len(mesh.nodes) * 3))
    inertia = 1
    for el in mesh.elements:
        node1 = el.node1
        node2 = el.node2

        dx = node1.x - node2.x
        dy = node1.y - node2.y
        L = np.sqrt((dx**2) + (dy**2))
        m = L * density

        mass_matrix[(node1.node_id*3)+0,(node1.node_id*3)+0] += m/2
        mass_matrix[(node2.node_id*3)+0,(node2.node_id*3)+0] += m/2
        mass_matrix[(node1.node_id*3)+1,(node1.node_id*3)+1] += m/2
        mass_matrix[(node2.node_id*3)+1,(node2.node_id*3)+1] += m/2

    for node in mesh.nodes:
        mass_matrix[(node.node_id*3)+2,(node.node_id*3)+2] += inertia

    return mass_matrix

M = get_mass_matrix(mesh, 1)
K = construct_global_stiffness_matrix(mesh)

alpha = 0.01
beta = 0.001

C = (alpha * M) + (beta * K)

$$K_{eff} = K + c_0M + c_1C$$
$$f_{eff,n+1} = f_{ext,n+1} + M (c_0u_n + c_2v_n + c_3a_n) + C (c_1u_n + c_4v_n + c_5a_n) $$
$$K_{eff}u_{n+1} = f_{eff,n+1}$$

$$c_0 = \frac{1}{\beta_N \Delta t^2}$$
$$c_1 = \frac{\gamma_N}{\beta_N \Delta t}$$
$$c_2 = \frac{1}{\beta_N \Delta t}$$
$$c_3 = \frac{1}{2\beta_N}-1$$
$$c_4 = \frac{\gamma_N}{\beta_N} - 1$$
$$c_5 = \Delta t (\frac{\gamma_N}{2\beta_N} - 1)$$

In [ ]:
DELTA_TIME = 0.1
BETA_N = 0.25
GAMMA_N = 0.5

C0 = (1 / (BETA_N * (DELTA_TIME ** 2)))
C1 = (GAMMA_N / (BETA_N * DELTA_TIME))
C2 = (1 / (BETA_N * DELTA_TIME))
C3 = (1 / (2 * BETA_N)) - 1
C4 = (GAMMA_N / BETA_N) - 1
C5 = DELTA_TIME * ((GAMMA_N / (2 * BETA_N) - 1))

In [ ]:
U = np.zeros(len(mesh.nodes) * 3)
V = np.zeros_like(U)
A = np.zeros_like(U)

print(C3)

In [ ]:
fixed_dofs = mesh.fixed_dofs
all_dofs = np.arange(len(mesh.nodes)*3)
free_dofs = np.setdiff1d(all_dofs, fixed_dofs)

U = np.zeros(len(mesh.nodes) * 3)
V = np.zeros_like(U)
A = np.zeros_like(U)

M_ff = M[np.ix_(free_dofs, free_dofs)]
C_ff = C[np.ix_(free_dofs, free_dofs)]
K_ff = K[np.ix_(free_dofs, free_dofs)]

loaded_dof = (10 * 3) + 1

def external_force(t):
    F = np.zeros(len(mesh.nodes) * 3)

    amplitude = -0.05
    angular_frequency = 0.1

    F[loaded_dof] = (
        amplitude * max((0.25 * np.sin(angular_frequency * t * 4)) + (np.cos(angular_frequency * t)),0)
    )

    return F

F0 = external_force(0.0)

initial_rhs = (
    F0[free_dofs]
    - C_ff @ V[free_dofs]
    - K_ff @ U[free_dofs]
)

A[free_dofs] = np.linalg.solve(
    M_ff,
    initial_rhs
)

A[fixed_dofs] = 0.0

K_eff_ff = (
    K_ff
    + C0 * M_ff
    + C1 * C_ff
)

In [ ]:
NUM_STEPS = 2_500
SAVE_EVERY = 10
DISPLAY_SCALE = 1.0

for step in range(NUM_STEPS):
    t_new = (step + 1) * DELTA_TIME

    F_new = external_force(t_new)

    # Effective force for the full system
    F_eff = (
        F_new
        + M @ (C0 * U + C2 * V + C3 * A)
        + C @ (C1 * U + C4 * V + C5 * A)
    )

    # Solve only for the free displacement DOFs
    U_new = np.zeros(len(mesh.nodes)*3)

    U_new[free_dofs] = np.linalg.solve(
        K_eff_ff,
        F_eff[free_dofs]
    )

    # Recover the new acceleration
    A_new = (C0 * (U_new - U) - C2 * V - C3 * A)

    # Recover the new velocity
    V_new = (V + DELTA_TIME * ((1.0 - GAMMA_N) * A + GAMMA_N * A_new))

    # Enforce stationary support conditions
    U_new[fixed_dofs] = 0.0
    V_new[fixed_dofs] = 0.0
    A_new[fixed_dofs] = 0.0

    # Advance the state
    U = U_new
    V = V_new
    A = A_new

    # Draw the current dynamic state, not the old static variable u
    if step % SAVE_EVERY == 0:
        plt.figure(figsize=(10, 5))

        # Original geometry
        mesh.draw(
            np.zeros_like(U),
            line_color="#AAAAAA",
            node_color="#AAAAAA",
        )

        stiffness = []
        for element in mesh.elements:
            dx = element.node1.x - element.node2.x
            dy = element.node1.y - element.node2.y
            L = np.sqrt(dx**2 + dy**2)

            dxu = dx - U[(element.node1.node_id * 3)] + U[(element.node2.node_id * 3)]
            dyu = dy - U[(element.node1.node_id * 3)+1] + U[(element.node2.node_id * 3)+1]
            Lu = np.sqrt(dxu**2 + dyu**2)

            axial_load = (Lu - L)
            stiffness.append(axial_load)

        stiffness = np.minimum((np.array(stiffness) / 0.1), 1)

        for i, element in enumerate(mesh.elements):
            u1x = U[(element.node1.node_id * 3)]
            u2x = U[(element.node2.node_id * 3)]
            u1y = U[(element.node1.node_id * 3) + 1]
            u2y = U[(element.node2.node_id * 3) + 1]
            cmap = plt.colormaps['plasma']
            plt.plot([element.node1.x + u1x, element.node2.x + u2x], [element.node1.y + u1y, element.node2.y + u2y], c=cmap(float(stiffness[i])))

        # mesh.draw(
        #     DISPLAY_SCALE * U,
        #     line_color="red",
        #     node_color="red"
        # )

        plt.xlim(-1, 13)
        plt.ylim(-1, 4)
        plt.gca().set_aspect("equal", adjustable="box")
        plt.title(f"Dynamic simulation: t = {t_new:.2f}")
        plt.tight_layout()

        frame_number = step // SAVE_EVERY

        f = -min(external_force(t_new))
        node_x = mesh.nodes[10].x + U[(10 * 3) + 0]
        node_y = mesh.nodes[10].y + U[(10 * 3) + 1]

        if f > 0.00001:
            plt.arrow(
                node_x, node_y + (f*10),            
                0, (-f*10),                          
                color="blue",
                width=0.05,
                head_width=0.2,
                head_length=0.15,
                length_includes_head=True,
                zorder=10
            )
       
 

        plt.savefig(
            f"frames/frame{frame_number:04d}.png",
            dpi=120
        )
        plt.close()

In [ ]:
!ffmpeg -y -framerate 30 -i frames/frame%04d.png -c:v libx264 -r 30 -pix_fmt yuv420p frames/animation.mp4
!ffmpeg -y -i frames/animation.mp4 -vf "fps=15,scale=640:-1:flags=lanczos" frames/animation.gif